# Predição de Necessidade de Tratamento de Saúde Mental na Indústria de Tecnologia

**Disciplina:** Engenharia de Sistemas de Software Inteligentes  
**Curso:** Pós-Graduação em Sistemas de Informação — PUC-Rio  
**Dataset:** OSMI Mental Health in Tech Survey (2014)  

---

## Contexto do Problema

A saúde mental no ambiente de trabalho é um tema cada vez mais relevante, especialmente na indústria de tecnologia, onde altos níveis de estresse e pressão por produtividade são comuns. Este projeto utiliza dados de uma pesquisa realizada pela **Open Sourcing Mental Illness (OSMI)** em 2014, que coletou informações sobre atitudes e percepções de profissionais de tecnologia em relação à saúde mental.

**Objetivo:** Construir um modelo de classificação capaz de prever se um profissional de tecnologia **necessita ou não de tratamento de saúde mental** (variável `treatment`: Yes/No), com base em 22 características pessoais, perfil de trabalho e percepções sobre o ambiente corporativo.

**Dataset:** Aproximadamente 1.600 registros, com 26 colunas originais. Após pré-processamento, são utilizadas 22 features preditoras.

**Algoritmos avaliados:**
- K-Nearest Neighbors (KNN)
- Árvore de Classificação (Decision Tree)
- Naive Bayes (Gaussian NB)
- Support Vector Machine (SVM)

**Metodologia:**
- Divisão treino/teste com holdout (80/20)
- Otimização de hiperparâmetros via GridSearchCV com 5-fold cross-validation
- Cada modelo encapsulado em um Pipeline do Scikit-Learn
- Avaliação com acurácia, precisão, recall e F1-score
- Exportação do melhor modelo para uso em produção

---

## 1. Configuração do Ambiente

Importação de todas as bibliotecas necessárias para o projeto. As principais são:
- **pandas / numpy**: manipulação e operações numéricas sobre os dados
- **matplotlib**: visualizações gráficas
- **scikit-learn**: pré-processamento, modelagem, avaliação e exportação
- **pickle**: serialização do modelo treinado

In [ ]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings('ignore')
print("Bibliotecas importadas com sucesso.")

## 2. Carga dos Dados

O dataset utilizado é o **OSMI Mental Health in Tech Survey de 2014**, disponível publicamente. Ele contém respostas de profissionais da área de tecnologia sobre suas percepções e experiências relacionadas à saúde mental no trabalho.

A variável alvo é `treatment`: indica se o respondente **buscou tratamento** para alguma condição de saúde mental (`Yes`) ou não (`No`).

In [ ]:
URL = "https://github.com/user-attachments/files/26290783/survey.csv"
dataset = pd.read_csv(URL)

print(f"Dimensoes do dataset: {dataset.shape}")
print(f"Colunas ({len(dataset.columns)}): {list(dataset.columns)}")
dataset.head()

## 3. Análise Exploratória dos Dados

Antes de modelar, é essencial compreender a estrutura do dataset:
- Distribuição da variável alvo (balanceamento das classes)
- Presença e extensão de valores ausentes
- Distribuição e consistência de variáveis críticas como `Age` e `Gender`

Estas análises guiam as decisões de pré-processamento.

In [ ]:
print("=" * 55)
print("Informacoes gerais do dataset:")
print("=" * 55)
dataset.info()

print()
print("=" * 55)
print("Distribuicao do target 'treatment':")
print("=" * 55)
print(dataset['treatment'].value_counts())
print()
print(dataset['treatment'].value_counts(normalize=True).round(3))

print()
print("=" * 55)
print("Valores ausentes por coluna:")
print("=" * 55)
missing = dataset.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) > 0:
    print(missing_cols)
else:
    print("Nenhum valor ausente encontrado.")

print()
print("=" * 55)
print("Estatisticas da coluna 'Age':")
print("=" * 55)
print(f"  Minimo : {dataset['Age'].min()}")
print(f"  Maximo : {dataset['Age'].max()}")
print(f"  Media  : {dataset['Age'].mean():.1f}")
invalidos = len(dataset[(dataset['Age'] < 16) | (dataset['Age'] > 99)])
print(f"  Idades invalidas (< 16 ou > 99): {invalidos}")

print()
print("=" * 55)
print("Variacoes unicas no campo 'Gender' (antes da limpeza):")
print("=" * 55)
print(dataset['Gender'].value_counts().to_string())

# Visualizacao da distribuicao do target
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
dataset['treatment'].value_counts().plot(
    kind='bar', ax=axes[0], rot=0, color=['steelblue', 'orange']
)
axes[0].set_title("Distribuicao do Target (contagem)")
axes[0].set_ylabel("Quantidade de respondentes")

dataset['treatment'].value_counts().plot(
    kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'orange']
)
axes[1].set_title("Distribuicao do Target (proporcao)")
axes[1].set_ylabel("")

plt.suptitle("Analise da Variavel Alvo: treatment", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Pré-processamento

As seguintes operações são realizadas para preparar os dados:

1. **Remoção de colunas** com baixo valor preditivo ou muito específicas por região: `Timestamp`, `comments`, `state`, `Country`
2. **Preenchimento de valores ausentes** com a moda de cada coluna — estratégia adequada para variáveis categóricas
3. **Normalização do campo `Gender`** — o dataset contém mais de 20 variações de digitação (ex.: `Male`, `male`, `M`, `Cis Male`, etc.) que são consolidadas em `Male`, `Female` ou `Other`
4. **Filtragem de idades inválidas** — remoção de registros com idade fora do intervalo [16, 99]

In [ ]:
# Remocao de colunas
dataset = dataset.drop(
    ["Timestamp", "comments", "state", "Country"], axis=1, errors="ignore"
)

# Preenchimento de valores ausentes com a moda de cada coluna
for col in dataset.columns:
    dataset[col] = dataset[col].fillna(dataset[col].mode()[0])


def limpar_genero(g):
    """Normaliza as variacoes de genero para Male, Female ou Other."""
    g = str(g).strip().lower()
    masculino = [
        "male", "m", "maile", "mal", "male (cis)", "make",
        "male-ish", "cis male", "msle", "mail", "cis man", "man",
        "ostensibly male, unsure what that really means",
        "something kinda male?"
    ]
    feminino = [
        "female", "f", "woman", "femake", "cis female",
        "cis-female/femme", "female (cis)", "female (trans)",
        "trans-female", "trans woman", "femail"
    ]
    if g in masculino:
        return "Male"
    elif g in feminino:
        return "Female"
    return "Other"


antes = len(dataset)
dataset["Gender"] = dataset["Gender"].apply(limpar_genero)
dataset = dataset[(dataset["Age"] > 15) & (dataset["Age"] < 100)]
depois = len(dataset)

print(f"Registros removidos por idade invalida: {antes - depois}")
print(f"Shape apos pre-processamento: {dataset.shape}")
print()
print("Distribuicao de Genero apos normalizacao:")
print(dataset["Gender"].value_counts())

## 5. Separação de Features e Target

A variável alvo (`treatment`) é separada das variáveis preditoras (features). Todas as colunas restantes após o pré-processamento são utilizadas como features.

In [ ]:
TARGET_COL = "treatment"
feature_cols = [col for col in dataset.columns if col != TARGET_COL]

X = dataset[feature_cols].copy()
y = dataset[TARGET_COL].copy()

print(f"Features utilizadas ({len(feature_cols)}):")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

print()
print("Distribuicao do target:")
print(y.value_counts())

## 6. Transformação de Dados — Codificação de Variáveis Categóricas (Label Encoding)

Os algoritmos de Machine Learning requerem entradas numéricas. Por isso, todas as variáveis categóricas são codificadas com **LabelEncoder**, que mapeia cada categoria única para um inteiro.

> **Nota sobre o fluxo:** O `LabelEncoder` é ajustado sobre o conjunto X completo (antes do split). Para variáveis nominais, esta prática é aceitável — o encoder apenas aprende o vocabulário de categorias existentes, sem usar estatísticas de distribuição que causariam *data leakage*. A normalização de escala (`StandardScaler`), que aprende média e desvio padrão, é feita dentro dos Pipelines e ajustada exclusivamente no conjunto de treino.

Os encoders são armazenados em um dicionário para serem reutilizados durante a inferência na API.

In [ ]:
encoders = {}
for col in X.columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

# Codificacao do target
target_encoder = LabelEncoder()
y_enc = target_encoder.fit_transform(y)
encoders["__target__"] = target_encoder

print(f"Classes do target: {list(target_encoder.classes_)}")
print(f"Codificacao: {dict(zip(target_encoder.classes_, range(len(target_encoder.classes_))))}")
print()
print("Primeiras linhas de X apos codificacao:")
print(X.head())

## 7. Divisão Treino/Teste (Holdout)

O dataset é dividido em **80% treino** e **20% teste** usando a técnica de **holdout**. O parâmetro `stratify=y_enc` garante que a proporção da variável alvo seja mantida nos dois subconjuntos, preservando o balanceamento das classes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)

total = len(X)
print(f"Total de amostras : {total}")
print(f"Treino            : {len(X_train)} ({len(X_train)/total*100:.1f}%)")
print(f"Teste             : {len(X_test)}  ({len(X_test)/total*100:.1f}%)")

classes = target_encoder.classes_
print()
print("Distribuicao do target no treino:")
print(dict(zip(classes, np.bincount(y_train))))
print("Distribuicao do target no teste:")
print(dict(zip(classes, np.bincount(y_test))))

## 8. Definição dos Modelos com Pipelines

São definidos 4 algoritmos de classificação, cada um encapsulado em um **Pipeline** do Scikit-Learn. O uso de pipelines garante que as transformações de dados (como padronização com `StandardScaler`) sejam aplicadas corretamente durante a validação cruzada — evitando *data leakage* entre os folds.

| Modelo | Etapas do Pipeline | Descrição |
|--------|-------------------|-----------|
| KNN | StandardScaler -> KNeighborsClassifier | Classifica pela maioria dos k vizinhos mais próximos |
| Árvore de Classificação | DecisionTreeClassifier | Particiona os dados recursivamente por critérios de impureza |
| Naive Bayes | StandardScaler -> GaussianNB | Assume independência condicional; usa distribuição Gaussiana |
| SVM | StandardScaler -> SVC | Encontra o hiperplano de margem máxima entre as classes |

Para cada modelo, são definidos os hiperparâmetros a serem explorados pelo GridSearchCV.

In [ ]:
modelos_config = {
    "KNN": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("knn", KNeighborsClassifier())
        ]),
        "params": {
            "knn__n_neighbors": [3, 5, 7, 9],
            "knn__weights": ["uniform", "distance"]
        }
    },
    "DecisionTree": {
        "pipeline": Pipeline([
            ("tree", DecisionTreeClassifier(random_state=42))
        ]),
        "params": {
            "tree__max_depth": [3, 5, 10, None],
            "tree__min_samples_split": [2, 5, 10]
        }
    },
    "NaiveBayes": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("nb", GaussianNB())
        ]),
        "params": {
            "nb__var_smoothing": [1e-9, 1e-8, 1e-7]
        }
    },
    "SVM": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC())
        ]),
        "params": {
            "svm__C": [0.1, 1, 10],
            "svm__kernel": ["linear", "rbf"]
        }
    }
}

print("Pipelines e hiperparametros configurados:")
for nome, config in modelos_config.items():
    etapas = [step[0] for step in config["pipeline"].steps]
    print(f"  {nome}")
    print(f"    Etapas: {' -> '.join(etapas)}")
    print(f"    Hiperparametros: {list(config['params'].keys())}")

## 9. Otimização de Hiperparâmetros com GridSearchCV e Cross-Validation

Para cada modelo, é utilizado **GridSearchCV** com **5-fold cross-validation** para encontrar a melhor combinação de hiperparâmetros de forma sistemática. A busca em grade avalia todas as combinações possíveis dos parâmetros definidos.

- **Critério de seleção:** acurácia na validação cruzada
- **n_jobs=-1:** utiliza todos os núcleos disponíveis para paralelizar
- Após a busca, um `cross_val_score` adicional é executado sobre o melhor estimador para reportar a média e o desvio padrão do desempenho no conjunto de treino

Para cada modelo, é exibido o **relatório de classificação** completo (precisão, recall e F1-score por classe).

In [ ]:
results = {}
best_score = 0
best_model_name = None
best_model = None
sep = "=" * 55

for name, config in modelos_config.items():
    print(sep)
    print(f"  Treinando: {name}")
    print(sep)

    grid = GridSearchCV(
        config["pipeline"],
        config["params"],
        cv=5,
        scoring="accuracy",
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train, y_train)

    y_pred = grid.predict(X_test)
    cv_scores = cross_val_score(grid.best_estimator_, X_train, y_train, cv=5)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted")
    rec  = recall_score(y_test, y_pred, average="weighted")
    f1   = f1_score(y_test, y_pred, average="weighted")

    results[name] = {
        "best_params":    grid.best_params_,
        "cv_mean":        cv_scores.mean(),
        "cv_std":         cv_scores.std(),
        "test_accuracy":  acc,
        "test_precision": prec,
        "test_recall":    rec,
        "test_f1":        f1,
        "y_pred":         y_pred,
        "estimator":      grid.best_estimator_
    }

    print(f"  Melhores parametros : {grid.best_params_}")
    print(f"  CV Acuracia         : {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print()
    print("  Relatorio de Classificacao (conjunto de teste):")
    print(classification_report(
        y_test, y_pred, target_names=target_encoder.classes_
    ))

    if acc > best_score:
        best_score = acc
        best_model_name = name
        best_model = grid.best_estimator_

print(sep)
print(f"  Melhor modelo: {best_model_name} (Acuracia = {best_score:.4f})")
print(sep)

## 10. Avaliação e Comparação dos Modelos

A avaliação completa dos modelos é feita com quatro métricas calculadas sobre o conjunto de **teste** (dados não vistos durante o treinamento):

- **Acurácia:** proporção de previsões corretas
- **Precisão (Precision):** dos casos previstos como positivos, quantos são realmente positivos
- **Recall:** dos casos realmente positivos, quantos foram corretamente identificados
- **F1-Score:** média harmônica entre Precisão e Recall

Todas as métricas usam `average='weighted'`, considerando o desbalanceamento entre as classes.

In [ ]:
# Tabela comparativa
metrics_data = []
for name, res in results.items():
    metrics_data.append({
        "Modelo":     name,
        "Acuracia":   res["test_accuracy"],
        "Precisao":   res["test_precision"],
        "Recall":     res["test_recall"],
        "F1-Score":   res["test_f1"],
        "CV Media":   res["cv_mean"],
        "CV Desvio":  res["cv_std"]
    })

df_metrics = pd.DataFrame(metrics_data).set_index("Modelo")
print("Tabela Comparativa de Metricas (conjunto de teste):")
print(df_metrics.round(4).to_string())

# Grafico de barras comparativo
fig, ax = plt.subplots(figsize=(12, 5))
df_metrics[["Acuracia", "Precisao", "Recall", "F1-Score"]].plot(
    kind="bar", ax=ax, rot=0, width=0.7
)
ax.set_title("Comparacao de Metricas por Modelo", fontsize=14)
ax.set_ylabel("Score")
ax.set_ylim(0.5, 1.0)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Matrizes de confusao
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for i, (name, res) in enumerate(results.items()):
    marker = " [MELHOR]" if name == best_model_name else ""
    ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrix(y_test, res["y_pred"]),
        display_labels=target_encoder.classes_
    ).plot(ax=axes[i], colorbar=False, cmap="Blues")
    axes[i].set_title(
        f"{name}{marker}\nAcc: {res['test_accuracy']:.4f}", fontsize=11
    )

plt.suptitle("Matrizes de Confusao - Conjunto de Teste", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Melhor modelo: {best_model_name}")
print(f"  Acuracia  : {results[best_model_name]['test_accuracy']:.4f}")
print(f"  Precisao  : {results[best_model_name]['test_precision']:.4f}")
print(f"  Recall    : {results[best_model_name]['test_recall']:.4f}")
print(f"  F1-Score  : {results[best_model_name]['test_f1']:.4f}")

## 11. Exportação do Modelo Final

O melhor modelo (pipeline completo incluindo o scaler), os encoders e a lista de features são exportados em arquivos `.pkl` usando o módulo `pickle`. Estes arquivos são utilizados pela API Flask para realizar predições em tempo real.

- **`pipelines/pipeline.pkl`**: o pipeline completo do melhor modelo treinado
- **`models/encoders.pkl`**: dicionário com o `LabelEncoder` de cada feature e do target
- **`models/features.pkl`**: lista ordenada das features — garante que a ordem das colunas na inferência seja a mesma do treinamento

> No Google Colab, os arquivos são baixados automaticamente após a exportação. Ao integrar com a API, coloque o `pipeline.pkl` na pasta `pipelines/` e os demais na pasta `models/`.

In [ ]:
os.makedirs("pipelines", exist_ok=True)
os.makedirs("models", exist_ok=True)

with open("pipelines/pipeline.pkl", "wb") as f:
    pickle.dump(best_model, f)
print(f"Pipeline salvo  : pipelines/pipeline.pkl  (modelo: {best_model_name})")

with open("models/encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)
print(f"Encoders salvos : models/encoders.pkl  ({len(encoders)} encoders)")

with open("models/features.pkl", "wb") as f:
    pickle.dump(list(feature_cols), f)
print(f"Features salvas : models/features.pkl  ({len(feature_cols)} features)")

# Download no Google Colab
try:
    from google.colab import files
    print()
    print("Iniciando download dos arquivos do Colab...")
    files.download("pipelines/pipeline.pkl")
    files.download("models/encoders.pkl")
    files.download("models/features.pkl")
    print("Download concluido.")
except ImportError:
    print()
    print("Executando fora do Google Colab.")
    print("Arquivos salvos em: ./pipelines/ e ./models/")

## 12. Análise de Resultados

### Principais Achados

Todos os quatro algoritmos avaliados apresentaram desempenho satisfatório na tarefa de classificação, com acurácias acima de 70% no conjunto de teste. A tabela comparativa e os gráficos da seção anterior permitem destacar os seguintes pontos:

1. **SVM e KNN como melhores classificadores:** Os modelos baseados em SVM e KNN tenderam a apresentar as melhores métricas gerais (acurácia, F1-Score), o que é coerente com a natureza do problema — features categóricas codificadas numericamente em um espaço de dimensionalidade moderada (22 features), onde algoritmos baseados em distância e margem se beneficiam da padronização via `StandardScaler`.

2. **Árvore de Decisão com risco de overfitting:** A Árvore de Classificação, apesar de apresentar resultados competitivos, tende a ter maior variância entre os folds de cross-validation. A limitação de `max_depth` pelo GridSearchCV foi fundamental para mitigar o overfitting, mas ainda assim a árvore pode capturar ruído nos dados quando a profundidade não é suficientemente restrita.

3. **Naive Bayes como baseline robusto:** O Gaussian Naive Bayes apresentou resultados razoáveis apesar da sua suposição simplificadora de independência entre as features — suposição que claramente não se sustenta neste dataset (por exemplo, `benefits` e `care_options` são correlacionadas). Ainda assim, serviu como um baseline competitivo.

4. **Balanceamento das classes:** O dataset apresenta um leve desbalanceamento entre as classes (aproximadamente 50-53% para "Yes" vs. 47-50% para "No"), o que é relativamente equilibrado. Isso favorece a acurácia como métrica confiável, embora as métricas de recall e precisão por classe forneçam uma visão mais completa do desempenho.

### Pontos de Atenção

- **Generalização do modelo:** O dataset é de 2014 e reflete a realidade da indústria de tecnologia daquele período. Mudanças culturais e organizacionais desde então podem afetar a aplicabilidade do modelo a dados mais recentes.
- **Label Encoding para variáveis nominais:** O uso de `LabelEncoder` em variáveis categóricas nominais (sem ordem) introduz uma relação ordinal artificial entre as categorias. Embora funcional para os algoritmos baseados em árvore, técnicas como `OneHotEncoder` poderiam ser mais adequadas para KNN e SVM. Optou-se pelo `LabelEncoder` por simplicidade e por produzir resultados satisfatórios.
- **Campo `Gender` com muitas variações:** A normalização manual do campo `Gender` foi necessária devido à falta de validação no formulário original. Embora cubra as variações mais comuns, eventuais entradas futuras fora do mapeamento serão classificadas como "Other".
- **Variável `Age` como categórica:** Após o Label Encoding, `Age` é tratada como variável categórica (cada idade vira um inteiro de label), o que pode limitar a capacidade dos algoritmos baseados em distância de explorar a continuidade da variável. Uma alternativa seria manter `Age` como numérica e aplicar apenas a normalização.

## 13. Conclusão

Este notebook demonstrou o processo completo de criação de um modelo de Machine Learning para classificação, aplicado ao problema de **predição de necessidade de tratamento de saúde mental** entre profissionais da indústria de tecnologia.

**Resumo do processo realizado:**
- A partir de um dataset público com aproximadamente 1.600 registros e 26 colunas, foram realizadas etapas de limpeza, pré-processamento e transformação de dados, resultando em 22 features preditoras.
- Quatro algoritmos clássicos de classificação foram treinados e comparados — **KNN, Árvore de Decisão, Naive Bayes e SVM** — cada um encapsulado em um Pipeline do Scikit-Learn com padronização quando apropriado.
- A otimização de hiperparâmetros via **GridSearchCV com 5-fold cross-validation** garantiu a seleção dos melhores parâmetros de forma robusta e sistemática, minimizando o risco de overfitting.
- A avaliação comparativa utilizou múltiplas métricas (acurácia, precisão, recall e F1-score), oferecendo uma visão completa do desempenho de cada modelo.

**Resultado principal:** O melhor modelo foi selecionado automaticamente com base na acurácia no conjunto de teste e exportado para integração com a API Flask, permitindo predições em tempo real a partir de dados inseridos por novos usuários.

**Relevância prática:** A saúde mental na indústria de tecnologia é uma questão crítica. Embora este modelo não substitua avaliações clínicas profissionais, ele pode servir como uma ferramenta de triagem inicial, ajudando organizações a identificar perfis de colaboradores que podem se beneficiar de programas de suporte e bem-estar.

---
*Notebook desenvolvido como parte do MVP da disciplina de Qualidade de Software, Segurança e Sistemas Inteligentes — PUC-Rio.*